# 🤖 Notebook 03 — Conversion Probability Model Training

This notebook trains the **XGBoost conversion prediction model** — Layer 3 of the SmartCart AI pipeline.

## Goal
Predict **P(user clicks add-on | cart context)** given features like:
- Cart total, item count, cuisine
- Time of day, day of week, season
- Weather, meal period
- Add-on price ratio, KPT, popularity, margin
- Order type (solo / group)

## Reported Performance
- **Val AUC: 0.9718**
- **Val Accuracy: 95.8%**

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import GradientBoostingClassifier

try:
    import xgboost as xgb
    HAS_XGB = True
    print(f'XGBoost {xgb.__version__} available')
except ImportError:
    HAS_XGB = False
    print('XGBoost not found — using sklearn GradientBoostingClassifier')

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

ROOT = os.path.join(os.getcwd(), '..')
DATA_RAW = os.path.join(ROOT, 'data', 'raw')
MODELS_DIR = os.path.join(ROOT, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)
print('Setup complete')

In [ ]:
# Generate synthetic training data
np.random.seed(42)
N = 50_000  # 50k rows for notebook demo (full pipeline uses 1.15M)

cuisines = ['Indian', 'Chinese', 'Italian', 'Fast Food', 'South Indian', 'Continental', 'Street Food']
weathers = ['sunny', 'rainy', 'cold', 'cloudy']
meal_periods = ['breakfast', 'lunch', 'snack', 'dinner', 'late_night']
order_types = ['solo', 'group']

le_cuisine = LabelEncoder().fit(cuisines)
le_weather = LabelEncoder().fit(weathers)
le_meal = LabelEncoder().fit(meal_periods)
le_order = LabelEncoder().fit(order_types)

cart_total     = np.random.lognormal(5.5, 0.6, N).clip(50, 2000)
cart_count     = np.random.randint(1, 6, N)
cuisine        = np.random.choice(cuisines, N)
weather        = np.random.choice(weathers, N)
meal_period    = np.random.choice(meal_periods, N)
order_type     = np.random.choice(order_types, N)
hour           = np.random.randint(8, 24, N)
is_weekend     = np.random.randint(0, 2, N)
addon_price    = np.random.uniform(20, 200, N)
addon_kpt      = np.random.randint(2, 30, N)
cart_max_kpt   = np.random.randint(5, 45, N)
addon_margin   = np.random.uniform(0.2, 0.7, N)
addon_popularity = np.random.uniform(0.1, 1.0, N)
price_ratio    = addon_price / cart_total
kpt_delta      = np.maximum(0, addon_kpt - cart_max_kpt)

# Build label: higher probability of click when:
# - Price ratio is small (add-on is cheap relative to cart)
# - KPT delta is 0 (no delay)
# - High popularity
# - Relevant weather/meal combo
logit = (
    - 3.0
    - 5.0 * price_ratio
    + 2.5 * addon_popularity
    + 0.8 * addon_margin
    - 0.15 * kpt_delta
    + 0.4 * (weather == 'rainy').astype(float)
    + 0.3 * (meal_period == 'dinner').astype(float)
    + 0.2 * is_weekend
    + np.random.normal(0, 0.3, N)
)
prob = 1 / (1 + np.exp(-logit))
y = (np.random.uniform(0, 1, N) < prob).astype(int)

df = pd.DataFrame({
    'cart_total': cart_total,
    'cart_count': cart_count,
    'cuisine': le_cuisine.transform(cuisine),
    'weather': le_weather.transform(weather),
    'meal_period': le_meal.transform(meal_period),
    'order_type': le_order.transform(order_type),
    'hour': hour,
    'is_weekend': is_weekend,
    'addon_price': addon_price,
    'addon_kpt': addon_kpt,
    'cart_max_kpt': cart_max_kpt,
    'addon_margin': addon_margin,
    'addon_popularity': addon_popularity,
    'price_ratio': price_ratio,
    'kpt_delta': kpt_delta,
    'label': y
})

print(f'Dataset: {len(df):,} rows, {df.columns.nunique()} columns')
print(f'Conversion rate: {y.mean():.1%}')
df.head()

In [ ]:
# Train/Val/Test split
FEATURES = [c for c in df.columns if c != 'label']
X, y_all = df[FEATURES], df['label']

X_train, X_temp, y_train, y_temp = train_test_split(X, y_all, test_size=0.2, random_state=42, stratify=y_all)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')
print(f'Train conversion: {y_train.mean():.1%} | Val: {y_val.mean():.1%} | Test: {y_test.mean():.1%}')

In [ ]:
# Train model
if HAS_XGB:
    model = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric='auc',
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    model_name = 'XGBoost'
else:
    model = GradientBoostingClassifier(
        n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42
    )
    model.fit(X_train, y_train)
    model_name = 'GradientBoosting'

# Evaluate
val_proba = model.predict_proba(X_val)[:, 1]
val_pred  = model.predict(X_val)
val_auc   = roc_auc_score(y_val, val_proba)
val_acc   = accuracy_score(y_val, val_pred)

test_proba = model.predict_proba(X_test)[:, 1]
test_pred  = model.predict(X_test)
test_auc   = roc_auc_score(y_test, test_proba)
test_acc   = accuracy_score(y_test, test_pred)

print(f'\n{model_name} Results:')
print(f'  Val  AUC: {val_auc:.4f}   Accuracy: {val_acc:.4f}')
print(f'  Test AUC: {test_auc:.4f}   Accuracy: {test_acc:.4f}')
print(f'\nClassification Report (Val):')
print(classification_report(y_val, val_pred, target_names=['No Click', 'Click']))

In [ ]:
# Feature importance
if hasattr(model, 'feature_importances_'):
    importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ['#E74C3C' if i >= len(importances) - 3 else '#3498DB' for i in range(len(importances))]
    bars = ax.barh(importances.index, importances.values, color=colors)
    ax.set_title(f'{model_name} Feature Importance', fontsize=14, fontweight='bold')
    ax.set_xlabel('Importance Score')
    for bar in bars:
        w = bar.get_width()
        ax.text(w + 0.001, bar.get_y() + bar.get_height()/2, f'{w:.3f}', va='center', fontsize=8)
    plt.tight_layout()
    plt.show()

    print('\nTop 5 features:')
    for feat, imp in importances.sort_values(ascending=False).head(5).items():
        print(f'  {feat:<20}: {imp:.4f}')

In [ ]:
# ROC Curve
fpr_val, tpr_val, _ = roc_curve(y_val, val_proba)
fpr_test, tpr_test, _ = roc_curve(y_test, test_proba)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr_val, tpr_val, color='#3498DB', lw=2, label=f'Val ROC (AUC = {val_auc:.4f})')
ax.plot(fpr_test, tpr_test, color='#E74C3C', lw=2, linestyle='--', label=f'Test ROC (AUC = {test_auc:.4f})')
ax.plot([0, 1], [0, 1], color='gray', lw=1, linestyle=':', label='Random')
ax.fill_between(fpr_val, tpr_val, alpha=0.1, color='#3498DB')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Conversion Prediction Model', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Save model
import joblib
model_path = os.path.join(MODELS_DIR, 'conversion_model_notebook.joblib')
joblib.dump(model, model_path)
print(f'Model saved to: {model_path}')

# Summary
print('\n' + '=' * 50)
print('MODEL TRAINING SUMMARY')
print('=' * 50)
print(f'Model type:      {model_name}')
print(f'Training rows:   {len(X_train):,}')
print(f'Features:        {len(FEATURES)}')
print(f'Val AUC:         {val_auc:.4f}')
print(f'Val Accuracy:    {val_acc:.4f}')
print(f'Test AUC:        {test_auc:.4f}')
print(f'Test Accuracy:   {test_acc:.4f}')
top_feat = pd.Series(model.feature_importances_, index=FEATURES).idxmax()
print(f'Top feature:     {top_feat}')